In [1]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent))

In [2]:
from src.dataset import ImageDataset
from torch.utils.data import DataLoader

annotations_file_trainval = Path("../data/preprocessed/trainval/annotations.csv")
img_dir_trainval = Path("../data/preprocessed/trainval/Images")

trainval_dataset = ImageDataset(annotations_file_trainval, img_dir_trainval)
trainval_dl = DataLoader(trainval_dataset, batch_size=32, shuffle=True)

In [3]:
from src.model import Model

model = Model()

In [39]:
from src.configs import S, B, C
from src.utils import convert_xywh_coords

def decode_preds(preds_batch):
    decoded_preds = []

    for pred in preds_batch:
        pred = pred.reshape((S, S, C + B * 5))
        objects = []

        for i in range(S):
            for j in range(S):
                pred_cell = pred[i][j]

                bbox_1 = convert_xywh_coords(pred_cell[20:24], i, j, False, True)
                bbox_2 = convert_xywh_coords(pred_cell[25:29], i, j, False, True)
                
                pred_1_confidence = (pred_cell[24].item(),)
                pred_2_confidence = (pred_cell[29].item(),)
                
                objects.append(bbox_1 + pred_1_confidence)
                objects.append(bbox_2 + pred_2_confidence)

        decoded_preds.append(objects)

    return decoded_preds

In [8]:
X_batch, y_batch = next(iter(trainval_dl))

X_batch.shape, y_batch.shape

(torch.Size([32, 3, 224, 224]), torch.Size([32, 7, 7, 30]))

In [11]:
preds = model(X_batch)
preds.shape

torch.Size([32, 1470])

In [16]:
preds = preds.reshape((preds.shape[0], S, S, B * 5 + C))
preds.shape

torch.Size([32, 7, 7, 30])

In [40]:
decoded_preds = decode_preds(preds)
decoded_preds

[[(tensor(1.8489, grad_fn=<SubBackward0>),
   tensor(2.0537, grad_fn=<SubBackward0>),
   tensor(-1.0051, grad_fn=<AddBackward0>),
   tensor(-2.3060, grad_fn=<AddBackward0>),
   -0.010385105386376381),
  (tensor(-1.6582, grad_fn=<SubBackward0>),
   tensor(-1.2394, grad_fn=<SubBackward0>),
   tensor(0.4026, grad_fn=<AddBackward0>),
   tensor(0.7138, grad_fn=<AddBackward0>),
   0.0035673717502504587),
  (tensor(32.0062, grad_fn=<SubBackward0>),
   tensor(-0.6039, grad_fn=<SubBackward0>),
   tensor(32.0768, grad_fn=<AddBackward0>),
   tensor(0.5665, grad_fn=<AddBackward0>),
   -0.013088776730000973),
  (tensor(32.9661, grad_fn=<SubBackward0>),
   tensor(-0.3332, grad_fn=<SubBackward0>),
   tensor(30.9767, grad_fn=<AddBackward0>),
   tensor(1.1329, grad_fn=<AddBackward0>),
   -0.005824597552418709),
  (tensor(64.1509, grad_fn=<SubBackward0>),
   tensor(0.8106, grad_fn=<SubBackward0>),
   tensor(63.6318, grad_fn=<AddBackward0>),
   tensor(-1.2123, grad_fn=<AddBackward0>),
   -0.0146186500787